# Practice

In [12]:
! pip install python-dotenv --quiet


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import os
from dotenv import load_dotenv

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")

print("Environment variables loaded successfully!")

Environment variables loaded successfully!


In [14]:
! pip install openai --quiet


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

#Load environment variables
load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

#Initialize the client
client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=api_key,
)

print("Azure OpenAI client initialized successfully.")

Azure OpenAI client initialized successfully.


### Single Response

In [16]:
response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "Answer briefly and directly. Do not provide your reasoning."
        },
        {
            "role": "user",
            "content": "I am travelling to Gokul, Uttar Pradesh, India. What is the best time to visit? ANd what should I see?"
        }
    ],
    max_tokens=500,
    model=deployment,
)

print(response.choices[0].message.content)

**Best Time to Visit:**
October to March.

**What to See:**
- Shri Vitthalnath Ji Temple (main temple)
- Thakurani Ghat
- Chaurasi Khamba
- Brahmand Ghat
- Raman Reti
- Shri Gokulnath Ji Temple


### Streaming Response

In [17]:
response = client.chat.completions.create(
    stream = True,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "I am travelling to Gokul, Uttar Pradesh, India. What is the best time to visit? ANd what should I see?"
        }
    ],
    max_tokens=2500,
    temperature=1.0,
    top_p=1.0,
    model=deployment,
)

for update in response:
    if update.choices:
        print(update.choices[0].delta.content or "", end="")

Gokul in Uttar Pradesh is a deeply significant pilgrimage town, intimately connected with the childhood of Lord Krishna. The "best" time to visit truly depends on what you're seeking—festivals, pleasant weather, or solitude.

Here’s a detailed guide to help you plan.

---

### Best Time to Visit Gokul

Gokul has a spiritual heartbeat that pulses strongest during its festivals. The weather is another major factor.

#### 1. For the Ultimate Festival Experience (Peak Season)
This is when Gokul is at its most vibrant, chaotic, and unforgettable. However, be prepared for massive crowds and premium accommodation prices.

- **Janmashtami (August/September):** This is the single most important time. The celebration of Krishna's birth is world-famous here, as Gokul was where he was secretly brought immediately after his birth. The entire town becomes a whirlwind of bhajans (devotional songs), dances, and dramatic reenactments of Krishna's childhood. The midnight celebrations are especially powe

## Multi-Turn Responses

In [18]:
response = client.chat.completions.create(
    stream = True,
    messages=[{"role":"system", "content":"You are a helpful assistant."},
              {"role":"user", "content":"I am travelling to Gokul. What should I see?"},
              {"role":"assistant", "content":"Gokul is a town in the Mathura district of Uttar Pradesh, India, known for its religious significance and historical sites. Here are some places you should consider visiting:Gokul Temple: This temple is dedicated to Lord Krishna and is a significant pilgrimage site.Nandgaon: A nearby village that is believed to be the childhood home of Lord Krishna.Radha Kund and Shyam Kund: Sacred ponds associated with the legends of Radha and Krishna. Govardhan Hill: A sacred hill that is associated with Lord Krishna's life and is a popular pilgrimage site.Yamuna River: Enjoy the scenic beauty of the Yamuna River, which flows through Gokul. Make sure to explore the local culture and cuisine as well!"},
              {"role":"user", "content":"What's so special about the Govardhan Hill?"},
    ],
    max_tokens=4096,
    temperature=1.0,
    top_p=1.0,
    model=deployment,
)

for update in response:
    if update.choices:
        print(update.choices[0].delta.content or "", end="")


Govardhan Hill is not just a geographical feature; it's one of the most deeply revered and theologically significant sites in the entire Vaisnava (devotion to Vishnu/Krishna) tradition. Its special nature stems from a single, defining pastime (lila) of Lord Krishna.

Here’s what makes it so special, broken down into the story, the theology, and the modern-day experience.

### 1. The Core Story: The Lifting of Govardhan Hill

The defining moment of Govardhan Hill is when Lord Krishna lifted it with the little finger of his left hand. Here’s the story:

- **Krishna's Divine Play:** As a young boy in Vrindavan, Krishna noticed his father, Nanda Maharaj, and the other cowherd men of the village preparing a grand annual sacrifice to appease Lord Indra, the rain god. They believed this was necessary for timely rains and good harvests.
- **Philosophical Challenge:** The young Krishna challenged this practice. He asked them to instead worship the Govardhan Hill, the cows, and the brahmins, arg

## Structured Outputs

In [29]:
import json
response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant. Give me the output in the following format: 1. <place> - <description>. Keep the description brief and to the point."
        },
        {
            "role": "user",
            "content": "I am travelling to Gokul, Uttar Pradesh, India. What should I see?"
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "places",
            "schema": {
                "type": "object",
                "properties": {
                    "place1": {"type": "string"},
                    "place2": {"type": "string"},
                    "place3": {"type": "string"}
                },
                "required": ["place1", "place2", "place3"],
                "additionalProperties": False
            },
            "strict": True
        },
    },
    max_tokens=500,
    model=deployment,
)

output = response.choices[0].message.content
output = json.loads(output)
print(f"Place 1: {output['place1']}")
print(f"Place 2: {output['place2']}")
print(f"Place 3: {output['place3']}")


Place 1: Shri Thakur Rani Ghat
Place 2: Shri Brahmand Ghat
Place 3: Shri Chaurasi Khamba Temple


### Safe Refusals

In [30]:
from openai import AzureOpenAI, BadRequestError

try: 
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role":"system", "content":"You are a helpful assistant."},
            {"role":"user", "content":"Give me detailed instructions to make a bomb."}
        ],
    )
    print(response.choices[0].message.content)
except BadRequestError as e:   
    if hasattr(e, "error") and e.error is not None:
        try:
            error_details = json.loads(e.error)
            print("Error details:", error_details)
        except Exception:
            print("Raw error response:", e.response.text)
    else:
        print("Error:", str(e))

I'm sorry, but I cannot assist with this request. If you have concerns about safety or security, I encourage you to reach out to the appropriate local authorities or professionals who can help address those concerns properly.


### Tokenizer

In [33]:
! pip install transformers --quiet

In [35]:
! pip install ipywidgets

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)

   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------

In [36]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-V3")

text = "Krishna is a major deity in Hinduism, worshipped as the eighth avatar of Vishnu and also as the supreme God in his own right. He is known for his playful and loving nature, as well as his role in the epic Mahabharata, where he serves as a charioteer and guide to the Pandava prince Arjuna. Krishna's teachings, particularly those found in the Bhagavad Gita, emphasize devotion, righteousness, and the importance of fulfilling one's duties."

tokens = tokenizer.tokenize(text)
print(f"Tokens: {tokens}")

token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"Token IDs: {token_ids}")

Tokens: ['K', 'rish', 'na', 'Ġis', 'Ġa', 'Ġmajor', 'Ġdeity', 'Ġin', 'ĠHinduism', ',', 'Ġworshipped', 'Ġas', 'Ġthe', 'Ġeighth', 'Ġavatar', 'Ġof', 'ĠVish', 'nu', 'Ġand', 'Ġalso', 'Ġas', 'Ġthe', 'Ġsupreme', 'ĠGod', 'Ġin', 'Ġhis', 'Ġown', 'Ġright', '.', 'ĠHe', 'Ġis', 'Ġknown', 'Ġfor', 'Ġhis', 'Ġplayful', 'Ġand', 'Ġloving', 'Ġnature', ',', 'Ġas', 'Ġwell', 'Ġas', 'Ġhis', 'Ġrole', 'Ġin', 'Ġthe', 'Ġepic', 'ĠMah', 'ab', 'har', 'ata', ',', 'Ġwhere', 'Ġhe', 'Ġserves', 'Ġas', 'Ġa', 'Ġchari', 'ote', 'er', 'Ġand', 'Ġguide', 'Ġto', 'Ġthe', 'ĠPand', 'ava', 'Ġprince', 'ĠAr', 'j', 'una', '.', 'ĠKrishna', "'s", 'Ġteachings', ',', 'Ġparticularly', 'Ġthose', 'Ġfound', 'Ġin', 'Ġthe', 'ĠBhag', 'avad', 'ĠG', 'ita', ',', 'Ġemphasize', 'Ġdevotion', ',', 'Ġrighteousness', ',', 'Ġand', 'Ġthe', 'Ġimportance', 'Ġof', 'Ġfulfilling', 'Ġone', "'s", 'Ġduties', '.']
Token IDs: [45, 58087, 2720, 344, 260, 3631, 74501, 295, 86306, 14, 120218, 412, 270, 39293, 66455, 294, 80052, 15943, 305, 990, 412, 270, 45993, 2998, 295,

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

sentences = [
    "Krishna is a major deity in Hinduism, worshipped as the eighth avatar of Vishnu and also as the supreme God in his own right.",
    "He is known for his playful and loving nature, as well as his role in the epic Mahabharata, where he serves as a charioteer and guide to the Pandava prince Arjuna."
    ]

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

encoded_input = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
with torch.no_grad():
    model_output = model(**encoded_input)

sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

print("Sentence embeddings:")
for i, sentence in enumerate(sentences):
    print(f"Sentence: {sentence}")
    print(f"Embedding: {sentence_embeddings[i].numpy()}\n") 


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence embeddings:
Sentence: Krishna is a major deity in Hinduism, worshipped as the eighth avatar of Vishnu and also as the supreme God in his own right.
Embedding: [-3.08933742e-02  2.51069162e-02 -2.53037773e-02  4.35362160e-02
 -7.25944489e-02  5.04508428e-03  6.32515475e-02  3.08441203e-02
  6.43712804e-02 -1.46605503e-02  2.96052051e-04 -3.13168280e-02
 -5.67391398e-04  2.73591373e-02  3.98183316e-02 -6.01929985e-02
 -7.52233900e-03  8.74804147e-03 -2.88183093e-02 -8.69930536e-02
  1.31044229e-02  5.89282392e-03 -7.36295879e-02 -1.82413366e-02
 -8.58718064e-03  9.33106709e-03  7.39328191e-02 -1.13347448e-01
  1.88890379e-02 -1.93342716e-02 -2.57620681e-02 -7.33458847e-02
 -5.30504994e-02  1.07006073e-01 -1.26211166e-01  6.10940382e-02
 -5.51192462e-02  2.71021072e-02 -4.58843745e-02 -7.27687031e-02
  1.13613699e-02  5.03251329e-02 -2.97444724e-02 -1.20497085e-01
  5.65395504e-02 -1.64242554e-02 -9.03381184e-02 -5.17699197e-02
  1.78689584e-02 -3.66655551e-02 -2.04421245e-02  6.

In [2]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")
text_to_count = "Krishna is a major deity in Hinduism, worshipped as the eighth avatar of Vishnu and also as the supreme God in his own right."
token_integers = encoding.encode(text_to_count)
token_count = len(token_integers)

print(f"Original text: {text_to_count}")
print(f"Token integers: {token_integers}")
print(f"Token count: {token_count}")


Original text: Krishna is a major deity in Hinduism, worshipped as the eighth avatar of Vishnu and also as the supreme God in his own right.
Token integers: [82265, 819, 3458, 374, 264, 3682, 74490, 304, 36142, 2191, 11, 83078, 6586, 439, 279, 37477, 21359, 315, 88252, 9110, 323, 1101, 439, 279, 44222, 4359, 304, 813, 1866, 1314, 13]
Token count: 31
